In [5]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

def preprocess_linux_logs(log_path="C:/Users/prane/Downloads/Linux/Linux.log", output_dir="C:/Users/prane/Downloads/Linux/processed_data/", start_year=2024):
    """
    Preprocess Linux logs for kernel and driver fault detection with proper year handling.
    
    Args:
        log_path (str): Path to the Linux log file
        output_dir (str): Directory to save processed data
        start_year (int): Year when the logs start (June)
    """
    print(f"Starting preprocessing of {log_path}...")
    
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Step 1: Read the log file
    try:
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            log_lines = f.readlines()
        print(f"Successfully read {len(log_lines)} lines from {log_path}")
    except Exception as e:
        print(f"Error reading log file: {e}")
        return None
    
    # Step 2: Define regex patterns for Linux log format
    # Month Day Time Hostname Service: Message
    log_pattern = r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})\s+(\S+)\s+(\S+):\s+(.*)'
    
    # Refined patterns to identify kernel and driver failures
    kernel_driver_failure_patterns = [
        r'kernel panic',
        r'not syncing',
        r'Oops:',
        r'Call Trace:',
        r'BUG:',
        r'Out of memory',
        r'oom-killer',
        r'hung task',
        r'blocked for more than',
        r'segfault at',
        r'general protection fault',
        r'Unable to handle kernel',
        r'Machine check events?',
        r'driver.*(fail|error|fault|crash|timeout|reset)',
        r'module.*(fail|error|fault|crash|timeout|reset)'
    ]
    
    # For broader kernel and driver related log detection
    kernel_patterns = [
        r'kernel',
        r'driver',
        r'module',
        r'firmware',
        r'hardware',
        r'device',
        r'cpu',
        r'memory',
        r'disk',
        r'error',
        r'warning',
        r'fail',
        r'critical',
        r'panic'
    ]
    
    # Step 3: Parse log lines into structured format with proper year handling
    structured_logs = []
    current_year = start_year
    last_month = None
    
    for line in log_lines:
        line = line.strip()
        if not line:
            continue
        
        match = re.search(log_pattern, line)
        if match:
            timestamp_str, hostname, service, message = match.groups()
            
            # Extract month to handle year transitions
            month_match = re.match(r'(\w{3})\s+', timestamp_str)
            if month_match:
                current_month = month_match.group(1)
                # Handle year transition (Dec to Jan)
                if last_month == 'Dec' and current_month == 'Jan':
                    current_year += 1
                last_month = current_month
            
            # Try to parse the timestamp with the correct year
            try:
                timestamp = datetime.strptime(f"{current_year} {timestamp_str}", "%Y %b %d %H:%M:%S")
            except:
                timestamp = None
            
            # Check if this is kernel or driver related
            is_kernel_related = False
            matched_patterns = []
            
            for pattern in kernel_patterns:
                if re.search(pattern, message, re.IGNORECASE) or re.search(pattern, service, re.IGNORECASE):
                    is_kernel_related = True
                    matched_patterns.append(pattern)
            
            # Check for specific failure patterns
            is_failure = False
            for pattern in kernel_driver_failure_patterns:
                if re.search(pattern, message, re.IGNORECASE):
                    is_failure = True
                    break
            
            # Create log entry
            log_entry = {
                'timestamp': timestamp,
                'hostname': hostname,
                'service': service,
                'message': message,
                'is_kernel_related': is_kernel_related,
                'is_failure': is_failure,
                'matched_patterns': "|".join(matched_patterns) if matched_patterns else ""
            }
            
            structured_logs.append(log_entry)
    
    # Convert to DataFrame
    df = pd.DataFrame(structured_logs)
    
    # Step 4: Filter to keep only kernel and driver related logs
    kernel_logs = df[df['is_kernel_related']].copy()
    print(f"Found {len(kernel_logs)} kernel and driver related logs out of {len(df)} total logs")
    print(f"Of which {kernel_logs['is_failure'].sum()} are identified as failures")
    
    # Step 5: Extract additional features from the logs
    def extract_features(df):
        """Extract additional features from log messages for fault detection"""
        # Severity classification
        def classify_severity(message):
            if re.search(r'emergency|critical|alert|panic|fault', message, re.IGNORECASE):
                return 4  # Critical
            elif re.search(r'error|fail', message, re.IGNORECASE):
                return 3  # Error
            elif re.search(r'warning|warn', message, re.IGNORECASE):
                return 2  # Warning
            elif re.search(r'notice|info', message, re.IGNORECASE):
                return 1  # Info
            else:
                return 0  # Unknown
        
        df['severity'] = df['message'].apply(classify_severity)
        
        # Message length features
        df['message_length'] = df['message'].str.len()
        df['word_count'] = df['message'].str.split().str.len()
        
        # Improved fault type detection
        def detect_fault_type(message):
            message_lower = message.lower()
            
            # Kernel panic is highest priority
            if re.search(r'kernel panic|not syncing', message_lower):
                return 'kernel_panic'
                
            # Memory-related issues
            if re.search(r'out of memory|oom-killer|oom|memory allocation', message_lower):
                return 'memory'
                
            # Storage-related issues
            if re.search(r'disk|storage|i/o error|block|fs|filesystem', message_lower):
                return 'storage'
                
            # Driver/module issues
            if re.search(r'driver.*(fail|error|fault|crash|timeout|reset)|module.*(fail|error|fault|crash|timeout)', message_lower):
                return 'driver'
                
            # CPU/Processing issues
            if re.search(r'cpu|processor|core|hung task|blocked for more than', message_lower):
                return 'cpu'
                
            # Network-related issues
            if re.search(r'network|ethernet|wifi|connection', message_lower):
                return 'network'
                
            # Security issues
            if re.search(r'security|auth|permission|access', message_lower):
                return 'security'
                
            # Kernel bugs and exceptions
            if re.search(r'oops:|bug:|call trace:|segfault at|general protection fault|unable to handle kernel', message_lower):
                return 'kernel_bug'
                
            # Hardware-related issues
            if re.search(r'machine check events?|hardware error', message_lower):
                return 'hardware'
                
            # Default case
            return 'other'
        
        df['fault_type'] = df['message'].apply(detect_fault_type)
        
        # Technical value extraction
        df['has_hex_address'] = df['message'].str.contains(r'0x[0-9a-fA-F]+', regex=True)
        df['has_memory_value'] = df['message'].str.contains(r'\d+\s*[kKmMgG][bB]', regex=True)
        df['has_process_id'] = df['message'].str.contains(r'pid|process', regex=True)
        
        return df
    
    # Apply feature extraction
    kernel_logs = extract_features(kernel_logs)
    
    # Step 6: Create time series data by resampling
    if not kernel_logs.empty and 'timestamp' in kernel_logs.columns:
        # Remove rows with missing timestamps
        kernel_logs_valid_ts = kernel_logs.dropna(subset=['timestamp'])
        
        if not kernel_logs_valid_ts.empty:
            # Sort by timestamp
            kernel_logs_valid_ts = kernel_logs_valid_ts.sort_values('timestamp')
            
            # Set timestamp as index
            time_series_df = kernel_logs_valid_ts.set_index('timestamp')
            
            # Resample to hourly intervals for time-series analysis
            # Round mean values to integers for readability
            def safe_round_mean(x):
                m = x.mean()
                return int(round(m)) if not np.isnan(m) else 0
            hourly_counts = time_series_df.resample('1H').agg({
                'is_kernel_related': 'sum',
                'is_failure': 'sum',  # Count of actual failures
                'severity': 'max',
                'message_length': safe_round_mean,
                'word_count': safe_round_mean,
                'has_hex_address': 'sum',
                'has_memory_value': 'sum',
                'has_process_id': 'sum'
            })
            
            # Count fault types per hour
            fault_type_counts = pd.DataFrame()
            for fault_type in kernel_logs_valid_ts['fault_type'].unique():
                hourly_fault = time_series_df[time_series_df['fault_type'] == fault_type].resample('1H').size()
                fault_type_counts[f'fault_{fault_type}'] = hourly_fault
            
            # Fill NaN values with 0
            fault_type_counts = fault_type_counts.fillna(0)
            
            # Merge the hourly counts with fault type counts
            merged_time_series = pd.concat([hourly_counts, fault_type_counts], axis=1)
            merged_time_series = merged_time_series.fillna(0)
            
            print(f"Created time series with {len(merged_time_series)} time intervals")
        else:
            merged_time_series = None
            print("No valid timestamps found for time series creation")
    else:
        merged_time_series = None
        print("No timestamp data available for time series creation")
    
    # Step 7: Create visualizations of the processed data
    if merged_time_series is not None:
        plt.figure(figsize=(12, 6))
        merged_time_series['is_kernel_related'].plot()
        plt.title('Kernel-Related Log Events Over Time')
        plt.ylabel('Number of Events')
        plt.xlabel('Time')
        plt.grid(True)
        plt.savefig(os.path.join(output_dir, 'kernel_events_time_series.png'))
        plt.close()
        
        # Plot failures over time
        plt.figure(figsize=(12, 6))
        merged_time_series['is_failure'].plot(color='red')
        plt.title('Kernel and Driver Failures Over Time')
        plt.ylabel('Number of Failures')
        plt.xlabel('Time')
        plt.grid(True)
        plt.savefig(os.path.join(output_dir, 'kernel_failures_time_series.png'))
        plt.close()
        
        # Create a heatmap of feature correlations
        plt.figure(figsize=(12, 10))
        corr_matrix = merged_time_series.corr()
        sns.heatmap(corr_matrix, annot=False, cmap='coolwarm')
        plt.title('Feature Correlation Matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'correlation_heatmap.png'))
        plt.close()
        
        # Create a bar chart of fault types
        plt.figure(figsize=(10, 6))
        fault_counts = kernel_logs['fault_type'].value_counts()
        fault_counts.plot(kind='bar', color='skyblue')
        plt.title('Distribution of Fault Types')
        plt.ylabel('Count')
        plt.xlabel('Fault Type')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'fault_type_distribution.png'))
        plt.close()
    
    # Step 8: Save processed data to CSV files
    if not kernel_logs.empty:
        # Save to CSV (convert timestamp to string for CSV storage)
        kernel_logs_for_csv = kernel_logs.copy()
        if 'timestamp' in kernel_logs_for_csv.columns:
            kernel_logs_for_csv['timestamp'] = kernel_logs_for_csv['timestamp'].astype(str)
        
        kernel_logs_for_csv.to_csv(os.path.join(output_dir, 'linux_kernel_logs.csv'), index=False)
        print(f"Saved structured kernel logs to {os.path.join(output_dir, 'linux_kernel_logs.csv')}")
    
    if merged_time_series is not None:
        merged_time_series.to_csv(os.path.join(output_dir, 'linux_time_series.csv'))
        print(f"Saved time series data to {os.path.join(output_dir, 'linux_time_series.csv')}")
    
    # Step 9: Create sliding windows for sequence prediction
    if merged_time_series is not None and len(merged_time_series) > 24:
        # Use a window size of 12 hours to predict the next hour
        window_size = 12
        X = []
        y = []
        
        # Normalize the data for ML
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(merged_time_series)
        
        # Create sequences
        for i in range(len(scaled_data) - window_size):
            X.append(scaled_data[i:i+window_size])
            # Target is the is_failure count - change index if needed
            failure_idx = merged_time_series.columns.get_loc('is_failure')
            y.append(scaled_data[i+window_size, failure_idx])
        
        X = np.array(X)
        y = np.array(y)
        
        # Save the windowed data
        np.save(os.path.join(output_dir, 'X_windows.npy'), X)
        np.save(os.path.join(output_dir, 'y_target.npy'), y)
        print(f"Created and saved {len(X)} sliding windows for sequence prediction")
    
    print(f"Preprocessing complete! Results saved to {output_dir}")
    
    return {
        'kernel_logs': kernel_logs,
        'time_series': merged_time_series
    }

# Example usage
if __name__ == "__main__":
    # Path to the Linux log file (from Loghub or your system)
    log_path = "C:/Users/prane/Downloads/Linux/Linux.log"  # Update this path as needed
    
    # Process the logs with correct year handling for June-February timeline
    # Change start_year to match your log data (e.g., 2023 if logs are from June 2023-Feb 2024)
    processed_data = preprocess_linux_logs(
        log_path=log_path,
        output_dir="C:/Users/prane/Downloads/Linux/processed_data/",
        start_year=2024  # Set this to the year when the June logs started
    )


Starting preprocessing of C:/Users/prane/Downloads/Linux/Linux.log...
Successfully read 25567 lines from C:/Users/prane/Downloads/Linux/Linux.log
Found 18223 kernel and driver related logs out of 25407 total logs
Of which 10492 are identified as failures
Created time series with 6332 time intervals
Saved structured kernel logs to C:/Users/prane/Downloads/Linux/processed_data/linux_kernel_logs.csv
Saved time series data to C:/Users/prane/Downloads/Linux/processed_data/linux_time_series.csv
Created and saved 6320 sliding windows for sequence prediction
Preprocessing complete! Results saved to C:/Users/prane/Downloads/Linux/processed_data/
